In [1]:
!pip install pytorch-tabnet autogluon -q
print("✅ 설치 완료")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 26.1 MB/s eta 0:0

In [2]:
import torch
print("GPU 사용 가능?", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

from google.colab import drive
drive.mount('/content/drive')

GPU 사용 가능? True
GPU: Tesla T4
Mounted at /content/drive


In [4]:
# ===============================================================
# 개선판 6등 코드 (최종)
# 적용된 개선:
# 1. FE 1차 + X_01 제거
# 2. TabNet 5-fold + MacroF1
# 3. AutoGluon f1_macro + 5-fold + target int 수정
# 4. Hungarian → Argmax
# 5. OOF 저장 (나중에 가중치 시뮬레이션용)
# ===============================================================
import os, random, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric

# ===============================================================
# Config
# ===============================================================
SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 21
print(f"✅ Using device: {DEVICE}")

DATA_DIR = "/content/drive/MyDrive/머신러닝/data"  # ⚠️ 본인 경로 확인!
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models_improved")
os.makedirs(MODEL_DIR, exist_ok=True)

# ===============================================================
# ⭐ Feature Engineering (FE 1차 + X_01 제거)
# ===============================================================
def feature_engineering(df):
    df = df.copy()
    if 'X_01' in df.columns:
        df = df.drop(columns=['X_01'])

    sensors = [c for c in df.columns if c.startswith('X_')]

    df['row_mean']   = df[sensors].mean(axis=1)
    df['row_std']    = df[sensors].std(axis=1)
    df['row_min']    = df[sensors].min(axis=1)
    df['row_max']    = df[sensors].max(axis=1)
    df['row_range']  = df['row_max'] - df['row_min']
    df['row_skew']   = df[sensors].skew(axis=1)
    df['row_median'] = df[sensors].median(axis=1)

    core_sensors = ['X_19', 'X_37', 'X_40', 'X_11']
    df['core_mean'] = df[core_sensors].mean(axis=1)
    df['core_std']  = df[core_sensors].std(axis=1)
    df['core_max']  = df[core_sensors].max(axis=1)
    df['core_min']  = df[core_sensors].min(axis=1)

    df['X19_X37_ratio'] = df['X_19'] / (df['X_37'] + 1e-6)
    df['X19_X40_ratio'] = df['X_19'] / (df['X_40'] + 1e-6)
    df['X19_X11_ratio'] = df['X_19'] / (df['X_11'] + 1e-6)
    df['X19_X37_diff']  = df['X_19'] - df['X_37']
    df['X19_X40_diff']  = df['X_19'] - df['X_40']
    df['X37_X40_diff']  = df['X_37'] - df['X_40']

    return df

# ===============================================================
# ⭐ MacroF1 메트릭 (TabNet용)
# ===============================================================
class MacroF1(Metric):
    def __init__(self):
        self._name = "macro_f1"
        self._maximize = True
    def __call__(self, y_true, y_score):
        y_pred = np.argmax(y_score, axis=1)
        return f1_score(y_true, y_pred, average='macro')

# ===============================================================
# Dataset
# ===============================================================
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        if self.y is None: return self.X[idx]
        return self.X[idx], self.y[idx]

# ===============================================================
# MLP Model
# ===============================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2, act="relu"):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
    def forward(self, x):
        residual = x
        out = self.act_fn(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return self.act_fn(out + residual)

class MLPEncoderWithHead(nn.Module):
    def __init__(self, input_dim, act="relu", num_classes=NUM_CLASSES):
        super().__init__()
        self.act_fn = nn.ReLU(inplace=True) if act == "relu" else nn.GELU()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act),
            nn.Linear(256, 256), nn.BatchNorm1d(256), self.act_fn,
            ResidualBlock(256, act=act)
        )
        self.embedding = nn.Linear(256, 128)
        self.projection = nn.Sequential(nn.Linear(128, 128), self.act_fn, nn.Linear(128, 128))
        self.classifier = nn.Sequential(nn.LayerNorm(128), self.act_fn, nn.Linear(128, num_classes))
    def forward(self, x):
        h = self.backbone(x)
        emb = self.embedding(h)
        z = self.projection(F.normalize(emb, dim=1))
        logits = self.classifier(emb)
        return emb, z, logits

# ===============================================================
# Loss Functions
# ===============================================================
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__(); self.temperature = temperature
    def forward(self, features, labels):
        device = features.device; bs = features.shape[0]
        labels = labels.contiguous().view(-1,1)
        mask = torch.eq(labels, labels.T).float().to(device)
        anchor_dot = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot, dim=1, keepdim=True)
        logits = anchor_dot - logits_max.detach()
        logits_mask = torch.ones_like(mask) - torch.eye(bs, device=device)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True)+1e-12)
        return -(mask * log_prob).sum(1).div(mask.sum(1)+1e-12).mean()

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim, lambda_c=1.0):
        super().__init__()
        self.register_buffer('centers', torch.randn(num_classes, feat_dim))
        self.lambda_c = lambda_c
    def forward(self, emb, labels):
        return self.lambda_c * ((emb - self.centers[labels])**2).sum() / (2.0 * emb.size(0))
    @torch.no_grad()
    def update_centers(self, emb, labels, lr=0.5):
        for c in torch.unique(labels):
            mask = labels==c
            if mask.sum()==0: continue
            self.centers[c] = (1-lr)*self.centers[c] + lr*emb[mask].mean(dim=0)

# ===============================================================
# MLP Training
# ===============================================================
def evaluate_model(model, loader):
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, _, logits = model(xb)
            preds.append(torch.softmax(logits, dim=1).cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds, trues = np.vstack(preds), np.concatenate(trues)
    return f1_score(trues, preds.argmax(1), average="macro"), preds

def train_mlp_model(X, y, X_test, act="relu", epochs=100):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds = np.zeros((len(X), NUM_CLASSES))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔥 MLP-{act} Fold {fold}")
        train_loader = DataLoader(TabularDataset(X[tr_idx], y[tr_idx]),
                                  batch_size=256, shuffle=True, drop_last=True)
        val_loader = DataLoader(TabularDataset(X[val_idx], y[val_idx]),
                                batch_size=512, shuffle=False)
        test_loader = DataLoader(TabularDataset(X_test), batch_size=512, shuffle=False)

        model = MLPEncoderWithHead(input_dim=X.shape[1], act=act).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
        ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
        supcon = SupConLoss(0.1)
        center = CenterLoss(NUM_CLASSES, 128, 1.0).to(DEVICE)

        best_f1, best_state, best_oof = -1, None, None
        for ep in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                emb, z, logits = model(xb)
                loss = 0.5*supcon(z,yb) + 1.0*ce_loss(logits,yb) + 0.1*center(emb,yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                center.update_centers(emb.detach(), yb.detach())
            f1, val_pred = evaluate_model(model, val_loader)
            scheduler.step(1-f1)
            if ep % 20 == 0: print(f"  Ep {ep}: f1={f1:.4f}")
            if f1 > best_f1:
                best_f1, best_state, best_oof = f1, model.state_dict(), val_pred

        print(f"  ✅ Best F1: {best_f1:.4f}")
        fold_scores.append(best_f1)
        oof_preds[val_idx] = best_oof
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), f"{MODEL_DIR}/mlp_{act}_fold{fold}.pth")

        model.eval()
        all_test = []
        with torch.no_grad():
            for xb in test_loader:
                xb = xb.to(DEVICE)
                _, _, logits = model(xb)
                all_test.append(torch.softmax(logits, dim=1).cpu().numpy())
        test_preds += np.vstack(all_test) / 5

    print(f"\n📊 MLP-{act} 평균 OOF F1: {np.mean(fold_scores):.4f}")
    return test_preds, oof_preds

# ===============================================================
# ⭐ TabNet (5-fold + MacroF1)
# ===============================================================
def train_tabnet(X, y, X_test):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    test_preds = np.zeros((len(X_test), NUM_CLASSES))
    oof_preds = np.zeros((len(X), NUM_CLASSES))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔥 TabNet Fold {fold}")
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        params = dict(
            n_d=32, n_a=16, n_steps=3, gamma=1.9, lambda_sparse=1e-5,
            optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=0.009),
            mask_type="sparsemax", seed=SEED, verbose=20,
            device_name='cuda' if torch.cuda.is_available() else 'cpu'
        )
        model = TabNetClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  eval_metric=[MacroF1],
                  patience=30, max_epochs=200, batch_size=1024)

        oof_pred = model.predict_proba(X_val)
        f1 = f1_score(y_val, oof_pred.argmax(1), average='macro')
        fold_scores.append(f1)
        print(f"  ✅ Fold {fold} F1: {f1:.4f}")

        oof_preds[val_idx] = oof_pred
        test_preds += model.predict_proba(X_test) / 5

    print(f"\n📊 TabNet 평균 OOF F1: {np.mean(fold_scores):.4f}")
    return test_preds, oof_preds

# ===============================================================
# ⭐ AutoGluon (f1_macro + target int 수정)
# ===============================================================
def train_autogluon(train_df, test_df):
    from autogluon.tabular import TabularPredictor
    predictor = TabularPredictor(
        label='target',
        eval_metric='f1_macro',
        path=os.path.join(MODEL_DIR, "autogluon"),
        problem_type='multiclass'  # ⭐ 명시적으로 multiclass 지정
    ).fit(
        train_data=train_df,
        time_limit=3600,
        presets='best_quality',
        num_bag_folds=5,
        verbosity=2
    )
    preds_proba = predictor.predict_proba(test_df)
    return preds_proba.values

# ===============================================================
# Main Pipeline
# ===============================================================
print("\n" + "="*60)
print("📊 데이터 로드 & FE 적용")
print("="*60)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"원본: train {train.shape}, test {test.shape}")

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)
print(f"FE 후: train {train_fe.shape}, test {test_fe.shape}")

# X, y 분리 + 정규화
X = train_fe.drop(columns=['ID', 'target']).values
y = train_fe['target'].values.astype(int)  # ⭐ int 명시
X_test = test_fe.drop(columns=['ID']).values

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)
print(f"피처 수: {X.shape[1]}")

# 모델 학습
print("\n" + "="*60); print("🚀 1/4 TabNet 학습 (5-fold)"); print("="*60)
tab_test, tab_oof = train_tabnet(X, y, X_test)

print("\n" + "="*60); print("🚀 2/4 MLP-ReLU 학습"); print("="*60)
relu_test, relu_oof = train_mlp_model(X, y, X_test, act="relu")

print("\n" + "="*60); print("🚀 3/4 MLP-GELU 학습"); print("="*60)
gelu_test, gelu_oof = train_mlp_model(X, y, X_test, act="gelu")

print("\n" + "="*60); print("🚀 4/4 AutoGluon 학습"); print("="*60)

# ⭐ AutoGluon용 DataFrame (target int 명시)
fe_cols_no_target = [c for c in train_fe.columns if c not in ['ID', 'target']]
train_ag = pd.DataFrame(X, columns=fe_cols_no_target)
train_ag['target'] = y.astype(int)  # ⭐ int 명시
test_ag = pd.DataFrame(X_test, columns=fe_cols_no_target)

print(f"train_ag shape: {train_ag.shape}")
print(f"target dtype: {train_ag['target'].dtype}")  # int64여야 함!
print(f"target 클래스 수: {train_ag['target'].nunique()}")  # 21이어야 함!

ag_test = train_autogluon(train_ag, test_ag)

# ===============================================================
# OOF 저장 (나중에 가중치 시뮬레이션용)
# ===============================================================
print("\n" + "="*60); print("💾 OOF 저장"); print("="*60)
np.save(f'{DATA_DIR}/oof_tabnet.npy', tab_oof)
np.save(f'{DATA_DIR}/oof_relu.npy', relu_oof)
np.save(f'{DATA_DIR}/oof_gelu.npy', gelu_oof)
np.save(f'{DATA_DIR}/test_tabnet.npy', tab_test)
np.save(f'{DATA_DIR}/test_relu.npy', relu_test)
np.save(f'{DATA_DIR}/test_gelu.npy', gelu_test)
np.save(f'{DATA_DIR}/test_autogluon.npy', ag_test)
print("✅ OOF & Test 확률 저장 완료")

# 모델별 OOF F1 출력
print("\n" + "="*60); print("📊 모델별 OOF Macro-F1"); print("="*60)
print(f"TabNet:   {f1_score(y, tab_oof.argmax(1), average='macro'):.4f}")
print(f"MLP-ReLU: {f1_score(y, relu_oof.argmax(1), average='macro'):.4f}")
print(f"MLP-GELU: {f1_score(y, gelu_oof.argmax(1), average='macro'):.4f}")

# 앙상블 + Argmax 후처리
print("\n" + "="*60); print("⚖️ 앙상블 + Argmax 후처리"); print("="*60)
ensemble = 0.25*tab_test + 0.25*relu_test + 0.25*gelu_test + 0.25*ag_test
preds = ensemble.argmax(axis=1)  # ⭐ Hungarian 제거

# 제출 파일
submission = pd.DataFrame({"ID": test["ID"], "target": preds})
output_path = os.path.join(DATA_DIR, "submission_improved.csv")
submission.to_csv(output_path, index=False)
print(f"\n✅ Saved: {output_path}")
print(f"\n📊 클래스별 예측 개수:")
print(pd.Series(preds).value_counts().sort_index())

✅ Using device: cuda

📊 데이터 로드 & FE 적용
원본: train (21693, 54), test (15004, 53)
FE 후: train (21693, 70), test (15004, 69)
피처 수: 68

🚀 1/4 TabNet 학습 (5-fold)

🔥 TabNet Fold 0


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.33243 | val_0_macro_f1: 0.0924  |  0:00:01s
epoch 20 | loss: 0.89419 | val_0_macro_f1: 0.6581  |  0:00:19s
epoch 40 | loss: 0.68621 | val_0_macro_f1: 0.73307 |  0:00:34s
epoch 60 | loss: 0.49147 | val_0_macro_f1: 0.73998 |  0:00:51s
epoch 80 | loss: 0.45447 | val_0_macro_f1: 0.78519 |  0:01:07s
epoch 100| loss: 0.39118 | val_0_macro_f1: 0.75221 |  0:01:23s
epoch 120| loss: 0.3592  | val_0_macro_f1: 0.80903 |  0:01:39s
epoch 140| loss: 0.35847 | val_0_macro_f1: 0.78984 |  0:01:55s
epoch 160| loss: 0.33159 | val_0_macro_f1: 0.83706 |  0:02:11s
epoch 180| loss: 0.331   | val_0_macro_f1: 0.79229 |  0:02:28s

Early stopping occurred at epoch 189 with best_epoch = 159 and best_val_0_macro_f1 = 0.86445


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  ✅ Fold 0 F1: 0.8645

🔥 TabNet Fold 1


...

epoch 0  | loss: 3.30128 | val_0_macro_f1: 0.10097 |  0:00:00s
epoch 20 | loss: 0.87142 | val_0_macro_f1: 0.68003 |  0:00:16s
epoch 40 | loss: 0.66292 | val_0_macro_f1: 0.72501 |  0:00:33s
epoch 60 | loss: 0.4915  | val_0_macro_f1: 0.76199 |  0:00:49s
epoch 80 | loss: 0.41724 | val_0_macro_f1: 0.7937  |  0:01:05s
epoch 100| loss: 0.38827 | val_0_macro_f1: 0.82153 |  0:01:20s
epoch 120| loss: 0.38144 | val_0_macro_f1: 0.82842 |  0:01:36s
epoch 140| loss: 0.37971 | val_0_macro_f1: 0.81684 |  0:01:52s

Early stopping occurred at epoch 152 with best_epoch = 122 and best_val_0_macro_f1 = 0.85539


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  ✅ Fold 4 F1: 0.8554

📊 TabNet 평균 OOF F1: 0.8529

🚀 2/4 MLP-ReLU 학습

🔥 MLP-relu Fold 0
  Ep 0: f1=0.5915
  Ep 20: f1=0.7896
  Ep 40: f1=0.8225
  Ep 60: f1=0.8585
  Ep 80: f1=0.8639
  ✅ Best F1: 0.8675

🔥 MLP-relu Fold 1
  Ep 0: f1=0.5798
  Ep 20: f1=0.8220
  Ep 40: f1=0.8641
  Ep 60: f1=0.8663
  Ep 80: f1=0.8699
  ✅ Best F1: 0.8725

🔥 MLP-relu Fold 2
  Ep 0: f1=0.6108
  Ep 20: f1=0.8030
  Ep 40: f1=0.8594
  Ep 60: f1=0.8752
  Ep 80: f1=0.8737
  ✅ Best F1: 0.8779

🔥 MLP-relu Fold 3
  Ep 0: f1=0.6144
  Ep 20: f1=0.7686
  Ep 40: f1=0.8294
  Ep 60: f1=0.8247
  Ep 80: f1=0.8232
  ✅ Best F1: 0.8366

🔥 MLP-relu Fold 4
  Ep 0: f1=0.6022
  Ep 20: f1=0.8040
  Ep 40: f1=0.8194
  Ep 60: f1=0.8353
  Ep 80: f1=0.8378
  ✅ Best F1: 0.8416

📊 MLP-relu 평균 OOF F1: 0.8592

🚀 3/4 MLP-GELU 학습

🔥 MLP-gelu Fold 0
  Ep 0: f1=0.6002
  Ep 20: f1=0.7940
  Ep 40: f1=0.8474
  Ep 60: f1=0.8678
  Ep 80: f1=0.8743
  ✅ Best F1: 0.8772

🔥 MLP-gelu Fold 1
  Ep 0: f1=0.6039
  Ep 20: f1=0.7926
  Ep 40: f1=0.8321
  Ep 60: 

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.55/14.56 GB
Total GPU Memory:   Free: 14.55 GB, Allocated: 0.02 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       9.81 GB / 12.67 GB (77.4%)
Disk Space Avail:   61.59 GB / 112.64 GB (54.7%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked 


💾 OOF 저장
✅ OOF & Test 확률 저장 완료

📊 모델별 OOF Macro-F1
TabNet:   0.8541
MLP-ReLU: 0.8593
MLP-GELU: 0.8637

⚖️ 앙상블 + Argmax 후처리

✅ Saved: /content/drive/MyDrive/머신러닝/data/submission_improved.csv

📊 클래스별 예측 개수:
0      892
1      676
2      419
3     1030
4      736
5      449
6      714
7      460
8     1050
9      328
10     703
11     646
12    1217
13     691
14     716
15     859
16     613
17     714
18     683
19     695
20     713
Name: count, dtype: int64


In [ ]:
# 저장된 데이터 로드
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from scipy.optimize import linear_sum_assignment

DATA_DIR = '/content/drive/MyDrive/머신러닝/data'

# OOF & Test 확률
tab_oof = np.load(f'{DATA_DIR}/oof_tabnet.npy')
relu_oof = np.load(f'{DATA_DIR}/oof_relu.npy')
gelu_oof = np.load(f'{DATA_DIR}/oof_gelu.npy')

tab_test = np.load(f'{DATA_DIR}/test_tabnet.npy')
relu_test = np.load(f'{DATA_DIR}/test_relu.npy')
gelu_test = np.load(f'{DATA_DIR}/test_gelu.npy')
ag_test = np.load(f'{DATA_DIR}/test_autogluon.npy')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
y = train['target'].values.astype(int)

# Hungarian 함수
def compute_quota(probs):
    N, C = probs.shape
    base = N // C; rem = N % C
    quota = np.full(C, base, int)
    order = np.argsort(-probs.mean(0))
    quota[order[:rem]] += 1
    return quota

def balanced_assign_hungarian(probs, quota):
    eps = 1e-15
    N, C = probs.shape
    cols, col_to_class = [], []
    for c in range(C):
        q = int(quota[c])
        cost_c = -np.log(probs[:,c] + eps)
        cols.append(np.repeat(cost_c[:,None], q, axis=1))
        col_to_class.extend([c]*q)
    cost = np.concatenate(cols, axis=1)
    row_ind, col_ind = linear_sum_assignment(cost)
    pred = np.zeros(N, int)
    pred[row_ind] = np.array(col_to_class)[col_ind]
    return pred

# ============================================
# 실험 1: 다양한 앙상블 + Argmax/Hungarian 비교
# ============================================
print("=" * 70)
print("🧪 여러 조합 시뮬레이션 (Test 예측 기준, 클래스 분포로 추정)")
print("=" * 70)

experiments = {
    "4명 균등 + Argmax":      (0.25*tab_test + 0.25*relu_test + 0.25*gelu_test + 0.25*ag_test, False),
    "4명 균등 + Hungarian":   (0.25*tab_test + 0.25*relu_test + 0.25*gelu_test + 0.25*ag_test, True),
    "3명만(AG 제외) + Argmax": ((tab_test + relu_test + gelu_test)/3, False),
    "3명만(AG 제외) + Hungarian": ((tab_test + relu_test + gelu_test)/3, True),
    "AG만 단독":             (ag_test, False),
    "AG만 단독 + Hungarian":   (ag_test, True),
    "GELU 우대 (TabNet 0.15, ReLU 0.15, GELU 0.35, AG 0.35)":
        (0.15*tab_test + 0.15*relu_test + 0.35*gelu_test + 0.35*ag_test, False),
}

# Test에는 실제 정답 없으므로, 각 조합의 OOF로 비교
# OOF 3명만 (AutoGluon OOF는 저장 못 함)
print("\n📊 OOF 점수로 검증 (3명만 비교 가능, AutoGluon OOF 미저장)")
print("-" * 70)

oof_experiments = {
    "TabNet 단독":         tab_oof,
    "MLP-ReLU 단독":       relu_oof,
    "MLP-GELU 단독":       gelu_oof,
    "3명 균등":            (tab_oof + relu_oof + gelu_oof) / 3,
    "GELU 우대(0.2/0.3/0.5)": 0.2*tab_oof + 0.3*relu_oof + 0.5*gelu_oof,
    "TabNet 빼고(0.4/0.6)":  0.4*relu_oof + 0.6*gelu_oof,
}

for name, probs in oof_experiments.items():
    # Argmax
    preds_argmax = probs.argmax(axis=1)
    f1_argmax = f1_score(y, preds_argmax, average='macro')
    # Hungarian
    quota = compute_quota(probs)
    preds_hung = balanced_assign_hungarian(probs, quota)
    f1_hung = f1_score(y, preds_hung, average='macro')
    print(f"  {name:40s} | Argmax: {f1_argmax:.4f} | Hungarian: {f1_hung:.4f} | 차이: {f1_hung-f1_argmax:+.4f}")

🧪 여러 조합 시뮬레이션 (Test 예측 기준, 클래스 분포로 추정)

📊 OOF 점수로 검증 (3명만 비교 가능, AutoGluon OOF 미저장)
----------------------------------------------------------------------
